# 라우팅, 재검색

- routing - 질문을 보고 갈림길을 고른다.
- 재검색 : loop 다시 찾고, 적절하게 멈춘다.
- checkpointer : 대화 상태를 저장하고 복원

In [ ]:
# 웹 서치 API 사용
from tavily import TavilyClient
TavilyClient().search("랭그래프", max_results=3)

{'query': '랭그래프',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://velog.io/@ohback/LangGraph',
   'title': 'LangGraph란 무엇이고 언제 쓰면 좋을까?',
   'content': '여기서 등장하는 게 LangGraph다. 이름 그대로 그래프(노드/엣지)로 LLM 애플리케이션의 흐름을 설계한다.  \n LangChain이 블록(모델·프롬프트·툴)을 잘 조립하게 해주는 공구상자라면, LangGraph는 그 블록들이 언제, 어떤 조건으로, 어떤 상태를 들고 움직일지를 정밀하게 오케스트레이션하는 조감도 + 신호등에 가깝다.\n\n  \n\n## 그래서 LangGraph가 뭐냐면..\n\n> LangChain 에서 개발한 LangGraph는 복잡한 생성형 AI 에이전트 워크플로를 구축, 배포 및 관리하도록 설계된 오픈소스 AI 에이전트 프레임워크입니다. 사용자가 확장 가능하고 효율적인 방식으로 대규모 언어 모델 (LLM)을 생성, 실행 및 최적화할 수 있도록 지원하는 도구와 라이브러리 세트를 제공합니다. LangGraph는 그래프 기반 아키텍처의 강력한 기능을 활용하여 AI 에이전트 워크플로 의 다양한 구성 요소 간의 복잡한 관계를 모델링하고 관리합니다. [...] > LangGraph는 단일 에이전트의 한계를 뛰어넘는 다중 에이전트 시스템을 구축할 수 있게 해줍니다. 각 에이전트는 특정 목표를 향해 자율적으로 행동하며, 다른 에이전트들과 협업하여 복잡한 문제를 해결합니다. 이러한 에이전트는 복잡한 의사결정을 위한 사전 정보, 작업 단계를 단축하기 위해 도움이 되는 미리 정의된 도구들 등에 접근하여 활용할지 스스로 결정할 수 있습니다.\n\n \n\n###### 출처: \n\n  \n\n## 구성 요소 및 예시 살펴보기\n\nLangGraph는 기본적으로 에이전트 워크플로를 그래프로 모델링 하는데, 세

In [3]:
import os, operator
from typing import Literal, TypedDict, Annotated   # Literal=값을 정해진 몇 개로 제한, Annotated=reducer 표시(01에서 배움)
from pydantic import BaseModel, Field   # 구조화 출력의 '틀'을 만드는 도구(아래 Route에서 씀)
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver   # 상태를 메모리에 저장하는 checkpointer(2-5에서 씀)


FAST_MODEL = "openai:gpt-5.4-nano"   # 분류·재작성 (가벼움, 저비용)
MAIN_MODEL = "openai:gpt-5.6-luna"   # 답변 생성 (기본)
fast = init_chat_model(FAST_MODEL)
main = init_chat_model(MAIN_MODEL)

# 📋 붙여넣기 — 실습용 문서·웹·계산 도구
DOCS = [   # 회사·과정 규정을 흉내 낸 아주 작은 '문서 창고'. retrieve 노드가 여기서 키워드로 찾는다.
    "연차 유급휴가는 1년간 80퍼센트 이상 출근한 근로자에게 15일이 주어진다.",
    "과정 수료 기준은 출석률 80퍼센트 이상이다.",
    "취업지원 프로그램 신청은 매 학기 초 2주 안에 해야 한다.",
    "수업에서 만든 코드는 개인 저장소에 백업하는 것을 권장한다.",
    "점심 시간은 12시부터 1시까지이며 강의실 취식은 금지된다.",
]

from tavily import TavilyClient
_tavily = TavilyClient(os.environ["TAVILY_API_KEY"])   # 웹 검색 클라이언트. 키는 .env에서 읽는다


def web_search(query: str) -> str:
    r = _tavily.search(query, max_results=3)   # 상위 3건만
    # 결과에서 제목·URL만 뽑아 한 덩어리 문자열로 만든다(모델에게 근거로 넘기기 좋게).
    return "\n".join(f"- {x['title']}: {x['url']}" for x in r["results"])


import ast, operator as _op
# 계산기를 eval 없이 '안전하게' 만들기 위한 준비물: 허용할 연산만 골라 함수에 매핑해 둔다.
_OPS = {ast.Add: _op.add, ast.Sub: _op.sub, ast.Mult: _op.mul,
        ast.Div: _op.truediv, ast.USub: _op.neg, ast.Pow: _op.pow}


def safe_calc(expr: str):
    """+ - * / ** 와 괄호만 허용하는 안전한 계산기. eval 을 쓰지 않는다(임의 코드 실행 위험 차단)."""
    def _ev(n):   # 파이썬이 파싱한 수식 나무(AST)를 재귀로 직접 계산한다
        if isinstance(n, ast.Constant): return n.value                       # 숫자 그대로
        if isinstance(n, ast.BinOp):   return _OPS[type(n.op)](_ev(n.left), _ev(n.right))  # 2항 연산
        if isinstance(n, ast.UnaryOp): return _OPS[type(n.op)](_ev(n.operand))             # 단항(-x)
        raise ValueError("허용되지 않은 식")   # 함수 호출·이름 등은 여기서 막힌다
    return _ev(ast.parse(expr, mode="eval").body)

In [22]:
# 4가지 중에서 선택을 강제하게 하는 스키마.
class Route(BaseModel):
    datasource: Literal["docs", "web", "calc", "direct"] = Field(
        description=("docs=사내규정이나 근무휴가 같은 우리 문서에 있을 질문"
                     "web=웹 검색이 필요한 질문"
                     "calc=수학 수식 계산"
                     "direct=기타 도구나 근거가 필요없는 질문")
    )

# 그래프의 상태값
class S(TypedDict):
    question: str
    route: str  # 라우터가 고른 선택지 4가지 중 1개
    query: str  # 웹 검색에 쓸 단어나 문장
    hits: list  # 웹 검색을 찾은 문서들
    answer: str
    attempts: int    # 몇 번 시도할 것인가.
    stop_reason: str # 끝내는 이유
    log: Annotated[list, operator.add]  # 각 노드마다 로그 남김. reducer

# 라우터
def router(s: S):
    r = fast.with_structured_output(Route).invoke(
        f"다음 질문을 docs/web/calc/direct 중 하나로 분류해라.사용자 질문: {s['question']}"
    )
    return {"route": r.datasource, "query": s["question"], "attempts": 0,
            "log": [f"router: '{s['question']}' -> {r.datasource}"]}

In [23]:
MAX_ATTEMPTS = 2   # 문서 검색을 최대 몇 번까지 시도할지. 이 수를 넘으면 giveup으로 멈춘다.


def retrieve(s: S):
    # 아주 단순한 키워드 검색(실습용). 한 글자 토큰은 아무 데나 걸려 오탐이 나므로 2글자 이상만 본다.
    toks = [t for t in s["query"].split() if len(t) >= 2]
    hits = [d for d in DOCS if any(t in d for t in toks)]   # 토큰이 하나라도 든 문서를 모은다
    return {"hits": hits, "attempts": s["attempts"] + 1,     # 시도 횟수를 하나 올린다
            "log": [f"retrieve#{s['attempts']+1}: query={s['query']!r} hits={len(hits)}"]}


def rewrite(s: S):
    # 검색이 비었을 때, fast 모델에게 '핵심 명사만' 다시 뽑게 해 검색어를 바꾼다.
    q = fast.invoke(
        f"검색이 실패했다. 이 질문의 핵심 명사 1~2개만 공백으로 구분해 출력하라: {s['question']}"
    ).content.strip()
    return {"query": q, "log": [f"rewrite: query -> {q!r}"]}   # 바뀐 query로 다음 retrieve가 다시 돈다


def grade(s: S) -> Literal["generate", "rewrite", "giveup"]:
    # 판정 함수: retrieve 결과를 보고 셋 중 하나를 돌려준다(이 문자열이 add_conditional_edges의 키가 됨).
    if s["hits"]:                       return "generate"   # 찾았으면 답 생성으로
    if s["attempts"] >= MAX_ATTEMPTS:   return "giveup"     # 다 썼으면 포기로(무한 반복 방지)
    return "rewrite"                                        # 아직 여유 있으면 재작성으로


def generate(s: S):
    # 찾은 문서(hits)만 근거로 답을 만든다. chr(10)='\n' — 근거들을 줄바꿈으로 이어 붙인다.
    a = main.invoke(
        f"근거:\n{chr(10).join(s['hits'])}\n\n질문: {s['question']}\n근거에 있는 내용만으로 한국어로 짧게 답하라."
    ).content
    return {"answer": a, "stop_reason": "answered", "log": ["generate: 근거로 답 생성"]}


def giveup(s: S):
    # 근거를 못 찾았을 때. 지어내지 않고 못 찾았다고 정직하게 답하며 멈춘다.
    return {"answer": "관련 근거를 찾지 못해 답하지 않습니다.",
            "stop_reason": "max_attempts", "log": ["giveup: 근거 없음, 멈춤"]}

def web(s: S):
    found = web_search(s["question"])   # Tavily로 웹 검색(2-1에서 만든 함수)
    a = main.invoke(f"다음 웹 검색 결과를 참고해 질문에 한국어로 짧게 답하라.\n{found}\n질문: {s['question']}").content
    return {"answer": a, "hits": [found], "stop_reason": "web",
            "log": [f"web: Tavily 검색 {found.count(chr(10))+1}건"]}


def calc(s: S):
    # 질문에서 수식만 뽑아 계산한다. 실패하면 못 한다고 밝힌다(지어내지 않는다).
    expr = fast.invoke(f"다음에서 계산할 수식만 파이썬 문법으로 출력(설명 금지): {s['question']}").content.strip()
    try:
        val = safe_calc(expr)          # 2-1의 안전한 계산기로 계산
        a = f"{expr} = {val}"
        reason = "calc"
    except Exception:                  # 수식이 아니거나 허용 안 된 식이면
        a, reason = "수식을 계산할 수 없습니다.", "calc_fail"
    return {"answer": a, "stop_reason": reason, "log": [f"calc: {expr!r}"]}


def direct(s: S):
    # 근거가 필요 없는 상식·잡담은 모델이 바로 답한다.
    a = main.invoke(s["question"]).content
    return {"answer": a, "stop_reason": "direct", "log": ["direct: 모델이 바로 답"]}

In [24]:
g = StateGraph(S)

In [25]:
for name, fn in [("router", router), ("retrieve", retrieve), ("rewrite", rewrite),
                 ("generate", generate), ("giveup", giveup),
                 ("web", web), ("calc", calc), ("direct", direct)]:
    g.add_node(name, fn)


In [26]:
g.add_edge(START, "router")
g.add_conditional_edges("router", lambda s: s['route'],
                        {"docs": "retrieve",
                         "web" : "web",
                         "calc" : "calc",
                         "direct" : "direct"})
g.add_conditional_edges("retrieve", grade,
                        {'generate' : 'generate',
                         'rewrite' : 'rewrite',
                         'giveup' : 'giveup'})

In [27]:
g.add_edge("generate", END)
g.add_edge("giveup", END)
g.add_edge("web", END)
g.add_edge("calc", END)
g.add_edge("direct", END)

In [28]:
app = g.compile(checkpointer=InMemorySaver())

In [29]:
print(app.get_graph().draw_ascii())

                                           +-----------+                                            
                                           | __start__ |                                            
                                           +-----------+                                            
                                                  *                                                 
                                                  *                                                 
                                                  *                                                 
                                             +--------+                                             
                                           ..| router |....                                         
                                       ....  +--------+  ..........                                 
                                  .....               .      ..... ........                

In [30]:
import uuid
thread_id = f"q-{uuid.uuid4().hex[:8]}"
cfg = {"configurable" : {"thread_id" : thread_id, "recusion_limit" : 25}}
app.invoke({"question" : "11324+(232*3544)가 뭐야"} , config = cfg)

{'question': '11324+(232*3544)가 뭐야',
 'route': 'calc',
 'query': '11324+(232*3544)가 뭐야',
 'answer': '11324+(232*3544) = 833532',
 'attempts': 0,
 'stop_reason': 'calc',
 'log': ["router: '11324+(232*3544)가 뭐야' -> calc", "calc: '11324+(232*3544)'"]}

In [31]:
def ask(question, thread_id=None):
    thread_id = thread_id or f"q-{uuid.uuid4().hex[:8]}"   # 안 주면 질문마다 새 대화(랜덤 id)
    # configurable.thread_id = 이 실행이 '어느 대화'인지. recursion_limit = 무한 순환 방지용 상한.
    cfg = {"configurable": {"thread_id": thread_id}, "recursion_limit": 25}
    out = app.invoke({"question": question, "log": []}, cfg)   # log=[] 로 시작(reducer가 이어 붙일 빈 리스트)
    print(f"\nQ: {question}")
    print(f"  route={out['route']} | attempts={out.get('attempts')} | stop={out['stop_reason']}")
    for line in out["log"]:
        print("   ·", line)
    print("  A:", out["answer"][:150])
    return out

In [32]:
ask("2026년 8월 기준 랭그래프 최신 버전 알려줘")


Q: 2026년 8월 기준 랭그래프 최신 버전 알려줘
  route=web | attempts=0 | stop=web
   · router: '2026년 8월 기준 랭그래프 최신 버전 알려줘' -> web
   · web: Tavily 검색 3건
  A: 2026년 8월 기준, 랭그래프(LangGraph)의 최신 메이저 버전은 **v1.0**입니다.  
(제공된 검색 결과에서는 세부 패치 버전까지 확인되지 않습니다.)


{'question': '2026년 8월 기준 랭그래프 최신 버전 알려줘',
 'route': 'web',
 'query': '2026년 8월 기준 랭그래프 최신 버전 알려줘',
 'hits': ['- LangChain/LangGraph V1 업데이트 이후 신기능 활용하여 에이전트 제작 ...: https://www.youtube.com/watch?v=Ipa6JNbFq4g\n- LangChain/LangGraph V1 업데이트 이후 신기능 활용하여 에이전트 제작하기!: https://www.youtube.com/watch?v=Crxxy0_O8oQ\n- 2026년, 여전히 LangChain인가? AI 엔지니어가 랭체인 도입을 ... - 블로그: https://m.blog.naver.com/beyond-zero/224171603930'],
 'answer': '2026년 8월 기준, 랭그래프(LangGraph)의 최신 메이저 버전은 **v1.0**입니다.  \n(제공된 검색 결과에서는 세부 패치 버전까지 확인되지 않습니다.)',
 'attempts': 0,
 'stop_reason': 'web',
 'log': ["router: '2026년 8월 기준 랭그래프 최신 버전 알려줘' -> web", 'web: Tavily 검색 3건']}

In [ ]:
# 그래프를 사진으로 저장
from pathlib import Path
Path("graph.png").write_bytes(app.get_graph().draw_mermaid_png())

28052